### 1 - Load Dataset

In [1]:
import numpy as np
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex
import time

# load dataset yang sama untuk semua metode
X = np.load("../0_data/processed/features.npy").astype(np.float32)
meta = pd.read_csv("../0_data/processed/meta.csv")

assert len(X) == len(meta)

dim = X.shape[1]
query_idx = 10
query = X[query_idx].reshape(1, -1)

k = 20

print("Dataset shape:", X.shape)

Dataset shape: (232725, 9)


### 2 - Annoy

In [2]:
print("=== Annoy ===")

ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()

for i, vec in enumerate(X):
    ann_index.add_item(i, vec)

ann_index.build(10)

annoy_build_time = time.time() - start

start = time.time()
annoy_neighbors = ann_index.get_nns_by_vector(X[query_idx], k, include_distances=False)
annoy_query_time = time.time() - start

print("Build time:", annoy_build_time)
print("Query time:", annoy_query_time)

=== Annoy ===
Build time: 1.099400520324707
Query time: 0.0


### 3 - FAISS

In [3]:
import faiss

print("\n=== FAISS ===")

faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
faiss_build_time = time.time() - start

start = time.time()
D, I_faiss = faiss_index.search(query, k)
faiss_query_time = time.time() - start

print("Build time:", faiss_build_time)
print("Query time:", faiss_query_time)


=== FAISS ===
Build time: 0.01169276237487793
Query time: 0.0030105113983154297


### 4 - HNSW

In [4]:
print("\n=== HNSW ===")

hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()

hnsw_index.init_index(
    max_elements=len(X),
    ef_construction=100,
    M=16
)

hnsw_index.add_items(X)

hnsw_build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
hnsw_neighbors, _ = hnsw_index.knn_query(query, k=k)
hnsw_query_time = time.time() - start

print("Build time:", hnsw_build_time)
print("Query time:", hnsw_query_time)


=== HNSW ===
Build time: 4.518056631088257
Query time: 0.0


### 5 - Evaluasi Hasil Recal@K 

In [5]:
def recall_at_k(true, approx, k):
    return len(set(true).intersection(set(approx))) / k

recall_annoy = recall_at_k(I_faiss[0], annoy_neighbors, k)
recall_hnsw = recall_at_k(I_faiss[0], hnsw_neighbors[0], k)

print("Recall Annoy:", recall_annoy)
print("Recall HNSW:", recall_hnsw)

Recall Annoy: 0.95
Recall HNSW: 1.0


In [ ]:
results = pd.DataFrame({
    "Method": ["Annoy", "FAISS (Flat)", "HNSW"],
    "Build Time (sec)": [annoy_build_time, faiss_build_time, hnsw_build_time],
    "Query Time (sec)": [annoy_query_time, faiss_query_time, hnsw_query_time],
    "Recall@K": [recall_annoy, 1.0, recall_hnsw]
})

results

print(results.to_string(index=False))

      Method  Build Time (sec)  Query Time (sec)  Recall@K
       Annoy          1.099401          0.000000      0.95
FAISS (Flat)          0.011693          0.003011      1.00
        HNSW          4.518057          0.000000      1.00
